# RSNA Intracranial Aneurysm Detection - Image / Meta Data Preparation

Data loading, preprocessing, and the BiomedCLIP zero-shot ensemble signal have been moved into `src/data/dataset.py` and `src/models/biomedclip.py` respectively -- this notebook now just demonstrates and exercises them.

# Setup

In [ ]:
import sys
from pathlib import Path

# Resolve the project root (parent of notebooks/) robustly: Jupyter has
# no __file__, and CWD varies by how this notebook is launched (its own
# directory, the project root, ...). Try each candidate and use whichever
# one actually contains `src/`, instead of assuming Path.cwd().parent.
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
_project_root = next((c for c in _candidates if (c / "src").is_dir()), None)
if _project_root is None:
    raise RuntimeError(
        "Could not locate the `src/` package from any of "
        f"{[str(c) for c in _candidates]}. Run this notebook from the "
        "project root or notebooks/ directory, or add the project root to "
        "sys.path manually."
    )
sys.path.insert(0, str(_project_root))

import numpy as np
import pandas as pd
import torch
from PIL import Image

from src.config import ID_COL, INPUT_SERIES_PATH, INPUT_TRAIN_PATH

## EDA

In [ ]:
df = pd.read_csv(INPUT_TRAIN_PATH)

In [ ]:
[INPUT_SERIES_PATH / i for i in df[ID_COL].unique()[:5].tolist()]

In [ ]:
df[ID_COL].unique()[:5].tolist()

## Data Loaders / Processors

See `src/data/dataset.py` for `BrainAneurysmDataset` and `create_dataloaders` (DICOM loading, CT/MR normalization, resizing, caching).

In [ ]:
from src.data.dataset import BrainAneurysmDataset, create_dataloaders

train_loader, val_loader = create_dataloaders(
    batch_size=4, num_workers=4, val_split=0.2, random_seed=42
)
print("\n=== Testing DataLoader ===")
batch = next(iter(train_loader))
print("volume:", batch["volume"].shape)       # [B,1,64,64,64]
print("image stack:", batch["image"].shape)   # [B,64,1,64,64]
print("labels:", batch["labels"].shape)       # [B, NUM_LABELS]
print("uid[0]:", batch["series_uid"][0])
if "segmentation" in batch:
    print("segmentation:", batch["segmentation"].shape)  # [B,64,64,64]
# convert first slice to PIL (if you want)
piller = BrainAneurysmDataset.to_pil_slices(
    {k: v[0] for k, v in batch.items() if isinstance(v, (list, dict, torch.Tensor))}
)

In [ ]:
sample = train_loader.dataset[0]

In [ ]:
sample.keys()

In [ ]:
sample["labels"].shape, sample["volume"].shape, sample["image"].shape

In [ ]:
sample["metadata"]

In [ ]:
# show every column (no truncation)
pd.set_option("display.max_columns", None)

# optional niceties:
pd.set_option("display.max_rows", 100)       # how many rows to show (None = unlimited)
pd.set_option("display.width", 200)          # line width for pretty printing
pd.set_option("display.max_colwidth", None)  # do not truncate long cell contents
pd.set_option("display.expand_frame_repr", False)  # keep wide dfs on one line if possible

## PubMedBERT / BiomedCLIP zero-shot ensemble

See `src/models/biomedclip.py`. https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224

In [ ]:
from src.models.biomedclip import (
    build_prompt_inputs,
    load_biomedclip_model,
    run_inference,
)

model, preprocess_train, preprocess_val, tokenizer, device = load_biomedclip_model()

In [ ]:
# [64, 1, 64, 64]

z = sample["image"].squeeze(1)
assert z.shape == torch.Size([64, 64, 64])

z_arr = z.numpy().reshape(1, -1)
assert tuple(z_arr.shape) == tuple((1, 262144))

z_out = preprocess_train(Image.fromarray(z_arr))
assert z_out.shape == torch.Size([3, 224, 224])

In [ ]:
prompt_inputs = build_prompt_inputs(tokenizer, device)

assert isinstance(prompt_inputs, torch.Tensor) and prompt_inputs.shape == torch.Size([6, 256])

In [ ]:
logits, sorted_indices = run_inference(model, z_out.unsqueeze(0).to(device), prompt_inputs)

In [ ]:
sample["volume"].shape  # [1, 64, 64, 64]

In [ ]:
sample["metadata"]

In [ ]:
sample["metadata"]["dicom_meta"]["ImageOrientationPatient"]

In [ ]:
sample["metadata"]["dicom_meta"]["ImagePositionPatient"]